# SI Figure S15: composite-formula ablation workbook (`ablations.xlsx`)

Rebuilds the `ablations.xlsx` workbook behind ~32 of SI Figure S15's images: ~25 composite-formula
variants (stationary geometry plus some combination of implicit PCM, explicit Desmond, and
rovibrational QCD corrections) across all 12 delta-22 solvents, at two reference levels:
DSD-PBEP86/pcSseg-3 (geometry PBE0/tz) and the MagNET-Zero training reference (WP04/pcSseg-2 for ¹H,
wB97X-D/pcSseg-2 for ¹³C).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import composite_models
import composite_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

def document_path(name):
    os.makedirs("documents", exist_ok=True)
    return os.path.join("documents", name)

# the shipped ablations.xlsx used 250 seeded train/test splits per (formula, solvent)
N_SPLITS = 250

In [ ]:
query_df_dft = delta22.add_composite_columns(delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False))
solutes = delta22.delta22_solutes(DELTA22_HDF5)
print(len(query_df_dft), "rows;", len(solutes), "solutes")

## Preview: one formula's mean test RMSE at the DSD-PBEP86 reference level

In [ ]:
# preview before the full build (all ~25 formulas x 12 solvents x 250 splits x 2 nuclei x 2
# reference levels)
level = composite_models.REFERENCE_LEVELS["dsd"]
preview = composite_models.ablation_rmse_table(query_df_dft, "H", level["method_h"], level["basis_h"],
                                 level["geometry_h"], solutes, n_splits=20)
preview[["chloroform", "benzene", "Mean Test RMSE"]].round(3)

## Build the full ablations workbook (both reference levels, both nuclei)

In [ ]:
output_path = document_path("ablations.xlsx")
composite_models.build_ablations_workbook(query_df_dft, solutes, output_path, n_splits=N_SPLITS)

## Correlations Between Features

Solvent-averaged Pearson r correlation matrix between the five composite-model features (stationary
shielding, PCM, Desmond, its vibrational analogue, QCD), both nuclei, plus a per-solvent
PCM-vs-Desmond table.

In [ ]:
for nucleus, label in [("H", "Proton"), ("C", "Carbon")]:
    corr = delta22.si_s15_feature_correlations(query_df_dft, nucleus)
    nuc_label = "1H" if nucleus == "H" else "13C"
    nuc_title = "$^{1}$H" if nucleus == "H" else "$^{13}$C"
    composite_plots.plot_feature_correlation_heatmap(
        corr["r"], vmin=-1, vmax=1, cmap="RdBu",
        title=f"Solvent-Averaged Pearson $r$ Correlation Matrix for {nuc_title}",
        save_path=figure_path(f"si_figure_s15_feature_corr_r_{nuc_label}.png"))
plt.show()

In [ ]:
pcm_desmond_corr = delta22.pcm_desmond_correlation_by_solvent(query_df_dft)
print("PCM vs. Desmond Pearson R by nucleus and solvent:")
display(pcm_desmond_corr.round(3))
composite_plots.plot_pcm_desmond_correlation_table(pcm_desmond_corr,
                                   save_path=figure_path("si_figure_s15_pcm_desmond_corr_table.png"))
plt.show()